# 共通論文の読解記録：朝と、説明し直した後

目的：AIを使わずに書いた自分の説明を保存する。準備の目安5分、読解時間は別。**このNotebookはAPIを呼びません。**
Colabでは次の準備セルのチェック欄をONにして実行します。コードを見る場合は「Show code」。
Colabはネット接続が必要。ローカルは準備済み環境を使います。

**スライドとの対応**：6「記録は手元に保存し、提出はPDFにする」、7「10分で読む：まず自分の説明を書く」。15:40の相互説明後も、このNotebookへ戻って自分の説明を残します。
朝はAIを開かず、分からない点もそのまま残します。午後はAI画面を閉じ、原論文を見ながら説明し直します。二つの段階は別の記録になり、朝の説明を上書きしません。
保存したMarkdownは自分が読むため、JSONLは全項目を再読込みするための形式です。


**この1ファイルで実行できます。** 他のNotebook・教材コード・授業設定の取得は不要です。上から順に実行します。準備用の定義セルは折りたたんであります。開くと普通のPythonとしてすべて読めます。
追加ライブラリとAPIキーは不要です。Pythonの標準ライブラリだけで記録・保存できます。


## 入力例 → 保存される内容

**練習用の記入例です。論文の正解や実際の学生の回答ではありません。** 朝はAIを使わず、自分が読めたこと・不明点に置き換えます。

1. 「記録する時点」で `record_stage = '朝：AIなしで読む'` を選びます。
2. 「自分の説明を書く」で次のように記入し、`SAVE = True` にします。

```python
annotations = {
    'student_explanation': '図の矢印で処理の順番を追ったが、各処理の違いはまだ説明できない。',
    'evidence_location': 'Paper2Agent、PDF p3、Fig.1a/b',
    'unresolved_point': '図のtoolsという語が、具体的に何を指すのか分からない。',
}
SAVE = True
```

**出力の見方**：`保存: (...)` に `.jsonl` と `.md` の2ファイルが表示され、`再読込み一致: True` になります。ファイル名の日時・識別番号は毎回変わります。Markdownには「本人の説明」「根拠の場所」「残る疑問」と上に書いた文章が入ります。AIの回答は生成しません。

**確認すること**：`.md` を開き、自分の3項目がそのまま残っていること。午後は `record_stage = '午後：説明し直す'` に変更して選択セルから実行し、改めて記入・保存します。朝のファイルは上書きされません。`None` は未記入または以前の文章を保持、`''` は空文字へ変更という意味です。

**使う関数**：`new_record` は空の記録を作る、`save_pair` は2形式で保存する、`load_records` は読み直す処理です。すべて下の準備セルに定義してあります。


In [ ]:
# 0. 保存先と必要なライブラリ（教材コードの取得は不要）
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()  # このNotebookを実行しているフォルダ
print('準備完了。授業設定と関数は以下のセルに含まれています。')

### 準備用の関数を定義する

以下も上から順に実行します。この段階ではAPI送信・モデル取得・回答の生成は行いません。折りたたみを開くと、記録、入力、送信などの関数を読めます。授業中に変更する欄は、この後の番号付きの手順にあります。

In [ ]:
# @title 準備：このNotebookで使うライブラリ
from __future__ import annotations
from copy import deepcopy
from datetime import datetime
from datetime import timezone
from html import escape
from math import isfinite
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from uuid import uuid4
import json
import re

In [ ]:
# @title 準備：読む論文・範囲・設問の設定
from __future__ import annotations
course = {'common_paper_id': 'P01',
 'day1_source_paper_id': 'P01',
 'common_reading': {'material_id': 'P01',
                    'source_version': 'Nature VoR 2026-09-16; publisher PDF retrieved 2026-09-30',
                    'assigned_scope': {'pdf_pages': [1, 3],
                                       'printed_pages': [1, 3],
                                       'text_sections': ['Abstract (p1)'],
                                       'figures': ['Fig.1a', 'Fig.1b'],
                                       'captions': ['Fig.1 caption (p3)'],
                                       'scope_id': 'P01_COMMON_v1'},
                    'question_set_id': 'Q_COMMON_3',
                    'scope_status': 'confirmed'},
 'homework_policy': {'status': 'confirmed',
                     'common_doi': '10.1038/s41586-026-11044-y',
                     'journal_titles': ['Nature', 'Science'],
                     'publication_status': 'published',
                     'article_type': 'research',
                     'open_access_required': True,
                     'different_from_common': True,
                     'selection_required': True,
                     'reviewer': 'student'},
 'activities': {'P0': {'material_choice': False, 'phase': 'R0', 'reading_ref': 'common_reading'},
                'P1': {'material_choice': False, 'phase': 'R1', 'reading_ref': 'common_reading'},
                'L1': {'material_choice': False, 'phase': None, 'reading_ref': 'common_reading'},
                'L2': {'material_choice': False, 'phase': None, 'reading_ref': 'common_reading'},
                'V1': {'material_choice': False, 'phase': None, 'material_id': 'V01'},
                'V2': {'material_choice': False, 'phase': None, 'material_id': 'V02'},
                'V3': {'material_choice': False, 'phase': None, 'material_id': 'V01'},
                'V4': {'material_choice': False, 'phase': None, 'material_id': 'V02'},
                'P2a': {'material_choice': False, 'phase': 'R2', 'reading_ref': 'common_reading'},
                'P2b': {'material_choice': False, 'phase': 'R3', 'reading_ref': 'common_reading'},
                'P3': {'material_choice': False, 'phase': 'R3', 'reading_ref': 'common_reading'},
                'D2T1': {'material_choice': False, 'material_id': 'T01', 'phase': 'transfer'},
                'D2T2': {'material_choice': False, 'material_id': 'T01', 'phase': 'transfer'},
                'D2W1': {'material_choice': False, 'material_id': None, 'phase': None},
                'D2W2': {'material_choice': False, 'material_id': None, 'phase': None},
                'HW1': {'material_choice': True, 'phase': 'homework'},
                'P01_REVIEW': {'material_choice': False, 'phase': 'homework', 'reading_ref': 'common_reading'},
                'TRANSFER': {'material_choice': False, 'material_id': 'P02', 'phase': 'transfer'},
                'METHODS': {'material_choice': False, 'material_id': 'M01', 'phase': None},
                'PRACTICE': {'material_choice': False, 'material_id': 'PRACTICE', 'phase': 'practice'}},
 'question_sets': {'Q_COMMON_3': ['この研究の主要な主張は何か。',
                                  'それを支える根拠は何か。指定した本文・図と対応付けて説明する。',
                                  'まだ理解できていない点、追加で確認したい点は何か。'],
                   'Q_METHODS_5': ['ツールを取り出す前に、元のtutorialを実行する目的は何か。',
                                   '自動テストは何を照合し、失敗が続いたツールをどう扱うか。',
                                   'AlphaGenomeの3条件の比較では、使える道具・情報がどう違うか。',
                                   '自由記述の回答は、誰がどのような基準で評価したか。',
                                   '論文の範囲外の質問を、どのように作り、何を評価したか。'],
                   'Q_TRANSFER_3': ['研究のどの工程を自動化し、人がどこに関与しているか。',
                                    '図1の処理の流れと評価結果を区別して説明できるか。',
                                    '今回の評価で示されたことと、まだ保証されていないことは何か。']}}

manifest = {'materials': [{'id': 'P01',
                'role': 'common_paper',
                'bibliography': {'authors': ['Jiacheng Miao',
                                             'Joe R. Davis',
                                             'Yaohui Zhang',
                                             'Jonathan K. Pritchard',
                                             'James Zou'],
                                 'title': 'Reimagining research papers as interactive and reliable AI agents',
                                 'journal': 'Nature',
                                 'year': 2026,
                                 'article_type': 'Article'},
                'doi': '10.1038/s41586-026-11044-y',
                'source_url': 'https://www.nature.com/articles/s41586-026-11044-y',
                'file_hash': '6727ef9ca72cfffb1e032a2f07c7b9a4a2b235154bdd779b135c3518433677bf',
                'adoption': 'adopted',
                'rights': {'view': 'confirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'},
                'open_access': True,
                'publication_status': 'published',
                'article_type': 'research',
                'license': {'id': 'CC-BY-NC-ND-4.0',
                            'url': 'https://creativecommons.org/licenses/by-nc-nd/4.0/',
                            'verified_from': 'publisher webpage Rights and permissions; PDF p9'}},
               {'id': 'V01',
                'role': 'vlm_case',
                'source_ref': 'P01',
                'assigned_scope': {'scope_id': 'P01_FIG2_v2',
                                   'pdf_pages': [3, 4, 5],
                                   'figures': ['Fig.2a', 'Fig.2b', 'Fig.2c', 'Fig.2d'],
                                   'captions': ['Fig.2 caption (p4)'],
                                   'text_sections': ['AlphaGenome agent: example and benchmark (p3 right column '
                                                     'to p5 opening)']}},
               {'id': 'V02',
                'role': 'vlm_case',
                'source_ref': 'P01',
                'assigned_scope': {'scope_id': 'P01_FIG12_CONTEXT_v1',
                                   'pdf_pages': [2, 3, 4, 5],
                                   'figures': ['Fig.1a', 'Fig.1b', 'Fig.2a', 'Fig.2b', 'Fig.2c', 'Fig.2d'],
                                   'captions': ['Fig.1 caption (p3)', 'Fig.2 caption (p4)'],
                                   'text_sections': ['Overview and MCP components and testing (p2)',
                                                     'AlphaGenome agent: example and benchmark (p3 right column '
                                                     'to p5 opening)']}},
               {'id': 'T01',
                'role': 'transfer_case',
                'bibliography': None,
                'doi': None,
                'source_url': None,
                'file_hash': None,
                'adoption': 'pending',
                'rights': {'view': 'unconfirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'},
                'source_version': None,
                'assigned_scope': None},
               {'id': 'P02',
                'role': 'transfer_paper',
                'bibliography': {'authors': ['Chris Lu', 'Cong Lu', 'Robert Tjarko Lange', 'et al.'],
                                 'title': 'Towards end-to-end automation of AI research',
                                 'journal': 'Nature',
                                 'year': 2026,
                                 'volume': 651,
                                 'pages': '914–919'},
                'doi': '10.1038/s41586-026-10265-5',
                'source_url': 'https://www.nature.com/articles/s41586-026-10265-5',
                'source_version': 'Nature Version of Record 2026-03-25',
                'adoption': 'adopted',
                'assigned_scope': {'text_sections': ['Abstract (PDF p1)',
                                                     'Generating manuscripts (PDF p2)',
                                                     'Human evaluation results (PDF p3–4)',
                                                     'Limitations (PDF p4–5)'],
                                   'figures': ['Fig.1a', 'Fig.1b', 'Fig.1c'],
                                   'pdf_pages': [1, 2, 3, 4, 5],
                                   'printed_pages': [914, 915, 916, 917, 918]},
                'question_set_id': 'Q_TRANSFER_3',
                'file_hash': 'a75e0d93447f400179136bf18d909df29e0c8ccaeba076a1dfb1beeef0e0e10d',
                'open_access': True,
                'publication_status': 'published',
                'article_type': 'research',
                'license': {'id': 'CC-BY-4.0', 'url': 'https://creativecommons.org/licenses/by/4.0/'},
                'rights': {'view': 'confirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'}},
               {'id': 'M01',
                'role': 'methods_case',
                'source_ref': 'P01',
                'assigned_scope': {'text_sections': ['Methods'], 'scope_id': 'P01_METHODS_DRAFT'},
                'question_set_id': 'Q_METHODS_5'},
               {'id': 'PRACTICE',
                'role': 'practice',
                'adoption': 'adopted',
                'bibliography': {'title': 'Notebookの操作練習（自作例。原論文ではない）'},
                'source_version': 'notebook-practice-2026-10-02',
                'assigned_scope': {'text_sections': ['このNotebookの入力例']},
                'rights': {'ai_input': 'confirmed'}}]}

In [ ]:
# @title 資料・設定を確認する関数：PHASES, ACTIVITY_NAMES, ValidationError, resolve_material, validate_homework_material
from __future__ import annotations
PHASES = {"P0": "R0", "P1": "R1", "P2a": "R2", "P2b": "R3", "P3": "R3"}


ACTIVITY_NAMES = {
    'PRACTICE': 'Notebookの操作練習（自作例）',
    'P0': '最初の読解（AIなし）', 'P1': '最初のAI読解',
    'L1': '言語モデルの仕組みを観察', 'L2': '質問方法の比較',
    'V1': '図2を画像で読む', 'V2': '画像の大きさを比べる',
    'P2a': '同じ3問への再回答', 'P3': '相互説明後の記録',
    'METHODS': 'Methodsの追加課題', 'TRANSFER': 'The AI Scientistへの応用',
    'HW1': 'Homework：別の原著論文', 'P01_REVIEW': 'Paper2Agentの追加読解',
}


class ValidationError(ValueError):
    """送信・配布前に修正が必要な設定。"""


def resolve_material(course: dict, manifest: dict, material_id: str) -> dict:
    """図の活動IDから原論文の版・権利を解決する。別論文への置換や循環を拒否。"""
    items = {m['id']: m for m in manifest['materials']}
    if material_id not in items:
        raise ValidationError('素材が登録されていません。')
    m = deepcopy(items[material_id])
    source_id = m.get('source_ref', material_id)
    if source_id != material_id:
        if source_id != course.get('day1_source_paper_id') or source_id not in items or items[source_id].get('source_ref'):
            raise ValidationError('第1日の図はP01を直接参照してください。')
        forbidden = {'bibliography', 'doi', 'source_url', 'source_version', 'file_hash', 'rights', 'adoption',
                     'pdf_url', 'pdf_page_count', 'license', 'open_access', 'publication_status', 'article_type'}
        if forbidden & m.keys():
            raise ValidationError('図の出典・版・権利・採用はP01から継承し、上書きしないでください。')
        m = {**deepcopy(items[source_id]), **m}
    if source_id == course['common_paper_id']:
        m['source_version'] = course['common_reading']['source_version']
        if material_id == source_id:
            m.update(deepcopy(course['common_reading']))
    m['source_material_id'] = source_id
    return m


def validate_homework_material(course: dict, material: dict) -> None:
    """掲載版・対象誌・OAの本人確認を検査。教員採用や科学的正しさは認定しない。"""
    policy = course.get('homework_policy', {})
    if policy.get('status') != 'confirmed':
        raise ValidationError('Homeworkの対象誌条件は教員確認待ちです。')
    doi_value = material.get('doi')
    doi = doi_value.strip().lower() if isinstance(doi_value, str) else ''
    common_doi = policy.get('common_doi', '').lower()
    if not common_doi:
        raise ValidationError('共通論文のDOI照合設定が必要です。')
    if material.get('id') == course['common_paper_id'] or doi == common_doi:
        raise ValidationError('HW1はP01と別論文を選んでください。P01継続はP01_REVIEWで記録できます。')
    bib = material.get('bibliography')
    if not isinstance(bib, dict) or not bib.get('title') or not bib.get('authors'):
        raise ValidationError('題名・著者・掲載誌を構造化した書誌に記入してください。')
    journal = bib.get('journal')
    if journal not in policy.get('journal_titles', []):
        raise ValidationError('Homeworkの対象誌リストに含まれていません。')
    if not re.fullmatch(r'10\.\d{4,9}/\S+', doi):
        raise ValidationError('掲載論文のDOIが必要です。')
    url = urlparse(material.get('source_url', ''))
    if url.scheme != 'https' or url.hostname not in {'www.nature.com', 'www.science.org'} or url.username or url.password:
        raise ValidationError('出版社の掲載論文ページを確認してください。')
    publisher = {'Nature': 'www.nature.com', 'Science': 'www.science.org'}
    if journal in publisher and url.hostname != publisher[journal]:
        raise ValidationError('掲載誌と出版社ページが一致しません。')
    if material.get('publication_status') != 'published' or material.get('article_type') != 'research':
        raise ValidationError('掲載済みの原著論文を選んでください。プレプリント・会議・ニュース等は対象外です。')
    if material.get('open_access') is not True:
        raise ValidationError('Open Access表示と利用条件を確認してください。')
    review = material.get('eligibility_review', {})
    if review.get('status') != 'confirmed' or not review.get('reviewer'):
        raise ValidationError('対象誌・原著・版・OAの確認状態と確認者を記録してください。')

In [ ]:
# @title 資料・設定を確認する関数：material_context
from __future__ import annotations
def material_context(course: dict, manifest: dict, activity_id: str,
                     selection: str | None = None, local_material: dict | None = None,
                     for_input: bool = True) -> dict[str, Any]:
    """共通課題はP01固定。Homeworkの学生選定だけ別のローカル定義を使う。"""
    if activity_id not in course["activities"]:
        raise ValidationError("活動IDを確認してください。")
    a = course["activities"][activity_id]
    if a.get("reading_ref") == "common_reading":
        mid = course["common_paper_id"]
        if selection is not None and selection != mid:
            raise ValidationError("共通課題の素材はP01に固定されています。")
    elif a.get("material_choice"):
        mid = selection
        if not mid:
            raise ValidationError("Homeworkの素材を明示してください。")
    else:
        mid = a.get("material_id")
        if selection is not None and selection != mid:
            raise ValidationError("この活動の素材は固定されています。")
    if mid and mid.startswith("U_"):
        if not a.get("material_choice") or not re.fullmatch(r"U_[A-Za-z0-9_-]+", mid):
            raise ValidationError("学生選定資料はHomework等で使用します。")
        m = deepcopy(local_material or {})
        if m.get("id") != mid or not all(m.get(k) for k in ("source_url", "bibliography", "source_version", "assigned_scope")):
            raise ValidationError("学生選定資料の出典・書誌・版・範囲を記入してください。")
        if for_input and (m.get("input_review", {}).get("status") != "confirmed" or not m.get("input_review", {}).get("reviewer")):
            raise ValidationError("入力の可否と確認者を記録してください。不明なら送らず教員へ相談してください。")
        m.update(role="student_paper", selection_origin="student", adoption="not_applicable", publication="not_granted")
    else:
        m = resolve_material(course, manifest, mid)
        if a.get("material_choice") and m["role"] not in {"common_paper", "extension_paper"}:
            raise ValidationError("HW1は条件を満たす教員の追加論文・学生選定論文を使います。")
        if for_input and (m.get("adoption") != "adopted" or m.get("rights", {}).get("ai_input") != "confirmed"):
            raise ValidationError("素材の採用またはAI入力の確認待ちです。")
        m["selection_origin"] = "common" if mid == "P01" else "instructor"
    if activity_id == 'HW1' and for_input:
        validate_homework_material(course, m)
    m["material_id"] = mid
    m["phase"] = a.get("phase")
    m["activity_id"] = activity_id
    return m

In [ ]:
# @title 記録を作り、保存・再読込みする関数：FIELDS, new_record, validate_record, _unique_records, _json_object, _reject_json_constant, _finite_json_float, _read_json, record_json
from __future__ import annotations
FIELDS = """session_id run_id activity_id phase timestamp material_id material_role selection_origin
source_version file_hash assigned_scope input_scope page_or_figure service access_route requested_model
reported_model conversation_mode prior_turn_count tool_usage input_kind prompt visible_system_instructions
parameters_requested parameters_confirmed response_raw status error_type retry_count usage cost_value cost_kind
elapsed_seconds student_explanation evidence_location verification_status peer_insight unresolved_point next_change
student_revision change_reason question_set_id input_review input_provenance error_message request_input""".split()


def new_record(context: dict, session_id: str | None = None) -> dict[str, Any]:
    """空の個人記録を作る。費用不明や該当なしはnullを維持する。"""
    r = dict.fromkeys(FIELDS)
    r.update(session_id=session_id or str(uuid4()), run_id=str(uuid4()), timestamp=datetime.now(timezone.utc).isoformat(),
             status="draft", cost_kind="unknown", retry_count=0, visible_system_instructions=[])
    for key in ("activity_id", "phase", "material_id", "selection_origin", "source_version", "assigned_scope", "question_set_id", "input_review"):
        r[key] = deepcopy(context.get(key))
    r["material_role"] = context.get("role")
    # 追加項目として保存し、旧記録の読込み互換性を保つ。
    r['source_material_id'] = context.get('source_material_id', context.get('material_id'))
    r['source_file_hash'] = context.get('file_hash')
    for key in ('eligibility_review', 'bibliography', 'doi', 'source_url', 'file_hash',
                'publication_status', 'article_type', 'open_access', 'license'):
        r[key] = deepcopy(context.get(key))
    return r


def validate_record(record: dict) -> None:
    """P01の段階とHomeworkの混在、費用の誤表示を防ぐ。"""
    if not isinstance(record, dict) or any(k not in record for k in FIELDS):
        raise ValidationError("記録の必須フィールドが不足しています。")
    aid, phase = record["activity_id"], record["phase"]
    if aid in PHASES and phase != PHASES[aid]:
        raise ValidationError("共通課題とR0～R3の対応が不正です。")
    if phase in {"R0", "R1", "R2", "R3"} and (record["material_id"] != "P01" or PHASES.get(aid) != phase):
        raise ValidationError("R0～R3は対応するP01共通課題専用です。")
    if aid == "HW1" and phase != "homework":
        raise ValidationError("HW1はhomework記録として保存してください。")
    fixed = {'METHODS': ('M01', None, 'Q_METHODS_5'), 'TRANSFER': ('P02', 'transfer', 'Q_TRANSFER_3')}
    if aid in fixed and (record['material_id'], phase, record['question_set_id']) != fixed[aid]:
        raise ValidationError('Methodsと別論文への応用は、それぞれ固定した素材・段階・設問で保存してください。')
    if phase == "R0" and (record["response_raw"] is not None or record["service"] not in {None, "none"}):
        raise ValidationError("R0はAIなしの記録です。")
    if record["cost_kind"] == "unknown" and record["cost_value"] is not None:
        raise ValidationError("費用不明を数値で埋めないでください。")
    if not all(isinstance(record[k], str) and record[k].strip() for k in ("session_id", "run_id")):
        raise ValidationError("記録IDが必要です。")
    for key in ("cost_value", "elapsed_seconds"):
        value = record[key]
        if value is not None and (type(value) not in (int, float) or not isfinite(value) or value < 0):
            raise ValidationError("費用・所要時間は非負の有限数またはnullにしてください。")


def _unique_records(records: list[dict]) -> None:
    """保存と再読込みの両方で、記録の配列とrunの一意性を確認する。"""
    if not isinstance(records, list):
        raise ValidationError("記録は配列として保存してください。")
    for record in records:
        validate_record(record)
    ids = [(r["session_id"], r["run_id"]) for r in records]
    if len(ids) != len(set(ids)):
        raise ValidationError("同じrunが重複しています。")


def _json_object(pairs: list[tuple]) -> dict:
    """同名フィールドが二つあるJSONを、後勝ちで読まない。"""
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValidationError("JSONのフィールド名が重複しています。元の記録を確認してください。")
        value[key] = item
    return value


def _reject_json_constant(value: str) -> None:
    """NaNやInfinityはJSONの数値として受け付けない。"""
    raise ValidationError("JSONに非標準の数値が含まれています。")


def _finite_json_float(value: str) -> float:
    """桁あふれをInfinityへ変換せず拒否する。"""
    number = float(value)
    if not isfinite(number):
        raise ValidationError("JSONの数値が大きすぎます。")
    return number


def _read_json(content: str) -> Any:
    """壊れたJSONを補修・欠落扱いせず、明示的に拒否する。"""
    try:
        return json.loads(content, object_pairs_hook=_json_object, parse_constant=_reject_json_constant,
                          parse_float=_finite_json_float)
    except json.JSONDecodeError:
        raise ValidationError("JSONを読み込めません。元の記録ファイルを確認してください。") from None


def record_json(value: Any, *, indent: int | None = None) -> str:
    """保存で型や項目が変わる値を、ファイル作成前に拒否する。"""
    try:
        content = json.dumps(value, ensure_ascii=False, indent=indent, allow_nan=False)
    except (TypeError, ValueError):
        raise ValidationError("記録にJSONで保存できない値があります。文字列・数値・null・リスト・辞書を使ってください。") from None
    if _read_json(content) != value:
        raise ValidationError("保存すると記録の型が変わります。辞書のキーは文字列、配列はリストにしてください。")
    return content

In [ ]:
# @title 記録を作り、保存・再読込みする関数：_readable_response, _text_block, save_records, load_records
from __future__ import annotations
def _readable_response(raw: Any) -> str:
    """壊れた応答も保存できるよう、表示可能な文章だけを一覧へ取り出す。"""
    if isinstance(raw, str):
        return raw
    if raw is None:
        return 'AIの応答なし'
    texts = []
    if isinstance(raw, dict) and isinstance(raw.get('output'), list):
        for item in raw['output']:
            if not isinstance(item, dict) or not isinstance(item.get('content'), list):
                continue
            for part in item['content']:
                if isinstance(part, dict) and part.get('type') == 'output_text' and isinstance(part.get('text'), str):
                    texts.append(part['text'])
    return '\n'.join(texts) if texts else '文章として表示できる応答がありません。状態と下の全項目を確認してください。'


def _text_block(value: Any) -> str:
    """質問・回答にMarkdownやHTMLがあっても、記録の構造を変えず原文として表示する。"""
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, indent=2)
    return '<pre>' + escape(text) + '</pre>'


def save_records(records: list[dict], path: str | Path) -> None:
    """明示操作で新しいファイルへ保存。同名上書きで初期記録を失わない。"""
    _unique_records(records)
    p = Path(path)
    if p.suffix not in {".jsonl", ".md"}:
        raise ValidationError("保存形式は.mdまたは.jsonlです。")
    if p.suffix == ".jsonl":
        content = "".join(record_json(r) + "\n" for r in records)
    else:
        lines = ["# 個人の読解記録", "", "生出力と本人の説明を区別。共有前に内容を確認してください。", ""]
        for number, r in enumerate(records, 1):
            cost = ('外部APIなし' if r['cost_kind'] == 'not_applicable' else
                    str(r['cost_value']) + ' USD' if r['cost_value'] is not None else '不明（0 USDとは扱わない）')
            # 記録の内部IDと、学生が読む活動名を分ける。
            activity = ACTIVITY_NAMES.get(r['activity_id'], str(r['activity_id']))
            model = r['reported_model'] or r['requested_model'] or ('AIなし' if r['service'] in {None, 'none'} else '不明')
            status = {'manual': '本人が記入', 'success': '応答完了', 'failed': '失敗',
                      'incomplete': '未完了', 'draft': '記入途中', 'pending': '処理中'}.get(r['status'], str(r['status']))
            mode = {'new': '新規', 'continue': '履歴を継続', None: '該当なし'}.get(r['conversation_mode'], str(r['conversation_mode']))
            lines += [f"## 記録 {number}: {escape(activity)}", "",
                      f"- 状態: {escape(status)}",
                      f"- モデル: {escape(str(model))}",
                      f"- 会話: {escape(mode)} / 過去の対話 {r['prior_turn_count'] or 0} 回",
                      f"- 費用: {cost}", ""]
            bibliography = r.get('bibliography')
            if isinstance(bibliography, dict) and bibliography.get('title'):
                lines += ['資料: ' + escape(str(bibliography['title'])), '']
            if r.get('error_message'):
                lines += ['### エラー・未完了の理由', '', _text_block(r['error_message']), '']
            for field, label in [('prompt', '送った質問'), ('input_scope', '送った資料の範囲'),
                                 ('student_explanation', '本人の説明'), ('evidence_location', '根拠の場所'),
                                 ('student_revision', '修正した説明'), ('unresolved_point', '残る疑問'),
                                 ('next_change', '次に変えること'), ('change_reason', '変更した理由')]:
                if r.get(field) is not None:
                    lines += ['### ' + label, '', _text_block(r[field]), '']
            lines += ['### AIの生回答', '', _text_block(_readable_response(r['response_raw'])), '']
        lines += ["## 再読込み用の全項目", "", '<details>', '<summary>入力・履歴・添付を含む全項目（再読込み時はこの部分を変更しない）</summary>', '',
                  "<!-- SEMINAR_RECORDS -->", "```json", record_json(records, indent=2),
                  "```", "<!-- END_SEMINAR_RECORDS -->", '', '</details>', '']
        content = "\n".join(lines)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("x", encoding="utf-8") as f:
        f.write(content)


def load_records(path: str | Path) -> list[dict]:
    """未知の追加フィールドを含め、そのまま再読込みする。"""
    p = Path(path)
    content = p.read_text(encoding="utf-8")
    if p.suffix == ".jsonl":
        records = [_read_json(s) for s in content.splitlines() if s.strip()]
    elif p.suffix == ".md":
        start = "<!-- SEMINAR_RECORDS -->\n```json\n"
        end = "\n```\n<!-- END_SEMINAR_RECORDS -->"
        if content.count(start) != 1 or content.count(end) != 1:
            raise ValidationError("再読込み用の記録は1か所にまとめてください。欠落・重複した範囲を無視して読みません。")
        if end not in content.split(start, 1)[1]:
            raise ValidationError("再読込み用の記録が見つかりません。元のMarkdownを確認してください。")
        records = _read_json(content.split(start, 1)[1].split(end, 1)[0])
    else:
        raise ValidationError("読込み形式は.mdまたは.jsonlです。")
    _unique_records(records)
    return records

In [ ]:
# @title 入力の表示と保存先を作る関数：activity_description, activity_view, save_pair
from __future__ import annotations
def activity_description(course: dict, manifest: dict, activity_id: str) -> str:
    """授業で読む範囲と問いを、日本語の短い案内として表示する。"""
    view = activity_view(course, manifest, activity_id)
    lines = [ACTIVITY_NAMES.get(activity_id, activity_id)]
    if 'choices' in view:
        return '\n'.join(lines + ['論文の対象条件と入力条件を確認して選びます。'])
    context = material_context(course, manifest, activity_id, for_input=False)
    bibliography = context.get('bibliography') or {}
    title = bibliography.get('title', '') if isinstance(bibliography, dict) else str(bibliography)
    lines += ['資料: ' + title, '版: ' + str(view.get('source_version'))]
    scope = view.get('assigned_scope') or {}
    if isinstance(scope, dict):
        for field, label in [('pdf_pages', 'PDFのページ'), ('text_sections', '本文'),
                             ('figures', '図'), ('captions', '図注')]:
            if scope.get(field):
                lines.append(label + ': ' + ' / '.join(map(str, scope[field])))
    else:
        lines.append('読む範囲: ' + str(scope))
    for number, question in enumerate(view.get('questions') or [], 1):
        lines.append(f'{number}. {question}')
    return '\n'.join(lines)


def activity_view(course: dict, manifest: dict, activity_id: str) -> dict:
    """共通課題は固定参照を表示し、Homeworkだけ選択肢を示す。"""
    a = course["activities"][activity_id]
    if a.get("material_choice"):
        return {"activity_id": activity_id, "choices": ["対象条件を満たす教員追加論文", "学生選定U_<local_id>"],
                "journal_policy": course.get('homework_policy'), "phase": a["phase"]}
    context = material_context(course, manifest, activity_id, for_input=False)
    return {"activity_id": activity_id, "material_id": context["material_id"], "source_version": context.get("source_version"),
            "assigned_scope": context.get("assigned_scope"), "adoption": context["adoption"],
            "questions": course["question_sets"].get(context.get("question_set_id"), []), "phase": a.get("phase")}


def save_pair(records: list[dict], directory: str | Path = "outputs") -> tuple[Path, Path]:
    """手元へ保存する。サーバへ自動転送しない。"""
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    paths = tuple(Path(directory) / (suffix + ext) for ext in (".jsonl", ".md"))
    for path in paths:
        save_records(records, path)
    return paths

## 1. 記録する段階と、共通の論文・範囲・3問を確認する

朝は`record_stage='朝：AIなしで読む'`。相互説明後は`'午後：説明し直す'`へ変更して、このセルを実行します。論文・版・範囲・3問は同じです。


In [ ]:
record_stage = '朝：AIなしで読む'  # 午後は '午後：説明し直す'
stages = {'朝：AIなしで読む': 'P0', '午後：説明し直す': 'P3'}
if record_stage not in stages:
    raise ValueError('記録する段階は、朝か午後のどちらかを指定してください。')
activity_id = stages[record_stage]
print(activity_description(course, manifest, activity_id))
if 'manual_records' not in globals():
    manual_records = {}
if activity_id not in manual_records:
    context = material_context(course, manifest, activity_id, for_input=False)
    manual_records[activity_id] = new_record(context)
offline_record = manual_records[activity_id]

## 2. 自分の説明を書き、保存する

`None`は前の記入を保持します。自分の文を`'自分の説明'`のように囲んで記入し、`SAVE=True`にします。複数行なら`"""`で前後を囲めます。
朝・午後の両方を記入済みなら、二つの記録を同じファイルへ保存します。保存するたびに別の名前になり、古い保存ファイルも残ります。


In [ ]:
annotations = {
    'student_explanation': None,  # 自分の説明： '...' の中に記入
    'evidence_location': None,    # 根拠のページ・図・本文
    'unresolved_point': None,     # まだ分からないこと
}
SAVE = False
if SAVE:
    if any(x is not None and not isinstance(x, str) for x in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    offline_record.update({k: v for k, v in annotations.items() if v is not None})
    offline_record.update(status='manual', service='none', cost_kind='not_applicable')
    paths = save_pair(list(manual_records.values()), ROOT / 'outputs')
    print('保存:', [str(p.relative_to(ROOT)) for p in paths])
    assert load_records(paths[0]) == list(manual_records.values())
    assert load_records(paths[1]) == list(manual_records.values())
    print('再読込み一致: True')

## 閉じる前に

ColabのFiles欄で教材フォルダ内の`outputs`からJSONLとMarkdownをダウンロードします。再起動すると、Pythonの変数は消えます。Colabのランタイムが削除されると保存ファイルも消えるため、朝の記録は保存後に手元へダウンロードしてください。[Colab公式FAQ](https://research.google.com/colaboratory/faq.html)
朝のAI利用は教員が案内するGUIで行います。APIで質問する際は[対話Notebook](01_dialogue_lab.ipynb)へ進みます。
